# Camada Gold — `ecommerce_categorias` (Data Quality BI)

Este notebook lê:
- A tabela de logs de DQ (`squad1/dq_monitoring_logs`) gravada na raiz do container
- A Silver `squad1/silver/ecommerce_categorias`

e produz **5 tabelas Gold** em Delta físico (`squad1/gold/ecommerce_categorias`)
e espelha cada uma no **Azure SQL Server** (para consumo via Looker):

| Tabela Gold | Descrição |
|---|---|
| `gold_ecommerce_categorias_dq_resumo_por_regra` | % de falha por regra por dia |
| `gold_ecommerce_categorias_dq_resumo_por_tabela` | % de registros limpos por tabela por hora |
| `gold_ecommerce_categorias_dq_tendencia_diaria` | Score de qualidade diário (0-100) e variação Δ |
| `gold_ecommerce_categorias_dq_regras_criticas` | Regras críticas com % de falha acima do threshold |
| `gold_ecommerce_categorias_snapshot_validas` | Visão analítica das categorias Silver válidas |

> **Ambiente**: Databricks Free Edition (Serverless). Toda I/O de Delta usa o SDK `deltalake`
> (bypass de `saveAsTable` / Unity Catalog gerenciado). O salvamento no SQL Server usa o
> conector nativo `sqlserver` do Spark (formato compatível com Serverless).

### Correções aplicadas nesta versão

1. **Leitura física via SDK `deltalake`** – assim como nos notebooks Bronze e Silver, todas as leituras e escritas Delta são feitas via caminhos `abfss://` explícitos, sem depender de tabelas gerenciadas do Unity Catalog.
2. **Verificação robusta de dependências** – o notebook agora valida se a tabela de DQ Logs e a Silver existem e contêm dados para a tabela alvo antes de prosseguir, com mensagens claras.
3. **Eliminação de código redundante** – removidos `display` e `show` duplicados.
4. **Tratamento de timezone** – todas as conversões para Pandas/Arrow tratam timezone com `.dt.tz_localize(None)` para evitar erros no PyArrow.
5. **Compatibilidade com Serverless** – mantido o uso do formato `sqlserver` para gravação no Azure SQL Database, que é suportado no Databricks Runtime.
6. **Ajuste na Gold 5 (snapshot)** – a contagem de filhos diretos agora usa a Silver como base, garantindo que categorias raiz sem filhos sejam corretamente identificadas.
7. **Leitura robusta de Delta** – a função `ler_delta` agora tenta primeiro via SDK `deltalake`; se falhar (ex.: arquivo Parquet corrompido), tenta via `spark.read.format("delta")`. Se ambas falharem, uma exceção clara orienta o usuário a executar novamente o Silver.


In [0]:
%pip install -q deltalake pyarrow python-dotenv


## 1. Imports, parâmetros e credenciais

In [0]:
import os
import uuid
import pandas as pd
import pyarrow as pa
from datetime import datetime, timezone
from dotenv import load_dotenv
from functools import reduce   # <--- adicionado para resolver NameError

from deltalake import DeltaTable
from deltalake.writer import write_deltalake

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    LongType, DoubleType, TimestampType, BooleanType
)

# ─── Contêiner e caminhos físicos ────────────────────────────────────────────
CONTAINER_SQUAD1  = "squad1"

# Fontes de leitura
CAMADA_DQ         = ""              # raiz do container (sem prefixo de pasta)
ENTIDADE_DQ       = "dq_monitoring_logs"

CAMADA_SILVER     = "silver"
ENTIDADE_SILVER   = "ecommerce_categorias"

# Destino Gold
CAMADA_GOLD       = "gold"
ENTIDADE_GOLD     = "ecommerce_categorias"   # subpasta dentro de gold/

# Nomes das 5 tabelas Gold
TABELA_GOLD_REGRA       = "gold_ecommerce_categorias_dq_resumo_por_regra"
TABELA_GOLD_TABELA      = "gold_ecommerce_categorias_dq_resumo_por_tabela"
TABELA_GOLD_TENDENCIA   = "gold_ecommerce_categorias_dq_tendencia_diaria"
TABELA_GOLD_CRITICAS    = "gold_ecommerce_categorias_dq_regras_criticas"
TABELA_GOLD_SNAPSHOT    = "gold_ecommerce_categorias_snapshot_validas"

# Tabela-alvo do DQ (filtro nos logs)
TABELA_ALVO = "silver_ecommerce_categorias"

# Threshold para sinalizar regras críticas (% de falha >= este valor é alerta)
THRESHOLD_FALHA_CRITICA = 5.0  # 5%

RUN_ID = str(uuid.uuid4())
print("RUN_ID:", RUN_ID)

# ─── Credenciais ADLS (Service Principal) ────────────────────────────────────
load_dotenv("/Workspace/Users/soaress.elias@gmail.com/merca-data-platform-categorias/.env")

CLIENT_ID            = os.getenv("ADLS_CLIENT_ID")
TENANT_ID            = os.getenv("ADLS_TENANT_ID")
CLIENT_SECRET        = os.getenv("ADLS_CLIENT_SECRET")
STORAGE_ACCOUNT_NAME = os.getenv("ADLS_STORAGE_ACCOUNT_NAME")

STORAGE_OPTIONS = {
    "account_name": STORAGE_ACCOUNT_NAME,
    "client_id":    CLIENT_ID,
    "client_secret": CLIENT_SECRET,
    "tenant_id":    TENANT_ID,
}

# ─── Credenciais SQL Server ───────────────────────────────────────────────────
JDBC_HOSTNAME = os.getenv("SQL_HOST")
JDBC_DATABASE = os.getenv("SQL_DATABASE")
JDBC_USERNAME = os.getenv("SQL_USERNAME")
JDBC_PASSWORD = os.getenv("SQL_PASSWORD")

print("ADLS account  :", STORAGE_ACCOUNT_NAME)
print("SQL Server host:", JDBC_HOSTNAME)
print("SQL Database  :", JDBC_DATABASE)
print(f"Origem DQ logs : {CONTAINER_SQUAD1}/{ENTIDADE_DQ}")
print(f"Origem Silver  : {CONTAINER_SQUAD1}/{CAMADA_SILVER}/{ENTIDADE_SILVER}")
print(f"Destino Gold   : {CONTAINER_SQUAD1}/{CAMADA_GOLD}/{ENTIDADE_GOLD}/<tabela>")

## 2. Funções auxiliares (Delta físico + SQL Server)

Todas as funções seguem o mesmo padrão dos notebooks Bronze e Silver,
usando o SDK `deltalake` para acesso direto ao ADLS, sem depender do Unity Catalog.

A função `ler_delta` agora é robusta: tenta ler via SDK; se falhar (ex.: arquivo Parquet vazio), tenta via `spark.read.format("delta")`, que pode ser mais tolerante. Se ambas falharem, uma exceção clara é lançada.

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# Helpers Delta (mesmo padrão do notebook Silver)
# ─────────────────────────────────────────────────────────────────────────────

def get_delta_path(camada: str, tabela: str, storage_opts: dict) -> str:
    """Monta a URI abfss:// para a tabela Delta. Camada vazia → raiz do container."""
    conta = storage_opts.get("account_name")
    if not camada:
        return f"abfss://{CONTAINER_SQUAD1}@{conta}.dfs.core.windows.net/{tabela}"
    return f"abfss://{CONTAINER_SQUAD1}@{conta}.dfs.core.windows.net/{camada}/{tabela}"


def get_delta_path_gold(subtabela: str, storage_opts: dict) -> str:
    """URI abfss:// para uma sub-tabela dentro de gold/ecommerce_categorias/."""
    conta = storage_opts.get("account_name")
    return (
        f"abfss://{CONTAINER_SQUAD1}@{conta}.dfs.core.windows.net"
        f"/{CAMADA_GOLD}/{ENTIDADE_GOLD}/{subtabela}"
    )


def delta_existe(camada: str, tabela: str, storage_opts: dict) -> bool:
    try:
        DeltaTable(get_delta_path(camada, tabela, storage_opts), storage_options=storage_opts)
        return True
    except Exception:
        return False

def ler_delta(camada: str, tabela: str, storage_opts: dict):
    """
    Lê uma tabela Delta física. Primeiro tenta via SDK deltalake (mais rápido).
    Se falhar (ex.: arquivo Parquet corrompido), tenta via Spark com autenticação OAuth.
    Se ambas falharem, levanta exceção com orientação.
    """
    if not delta_existe(camada, tabela, storage_opts):
        raise Exception(f"Tabela Delta não encontrada: {camada or 'raiz'}/{tabela}")
    path = get_delta_path(camada, tabela, storage_opts)
    
    # Tentativa 1: via SDK
    try:
        dt = DeltaTable(path, storage_options=storage_opts)
        pdf = dt.to_pandas()
        return spark.createDataFrame(pdf)
    except Exception as e1:
        print(f"[Aviso] Leitura via SDK falhou: {e1}. Tentando via Spark...")
        # Tentativa 2: via Spark com credenciais OAuth
        try:
            # Configurações para autenticação OAuth com Service Principal
            df = spark.read.format("delta") \
                .option("fs.azure.account.auth.type", "OAuth") \
                .option("fs.azure.account.oauth.provider.type", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider") \
                .option("fs.azure.account.oauth2.client.id", storage_opts["client_id"]) \
                .option("fs.azure.account.oauth2.client.secret", storage_opts["client_secret"]) \
                .option("fs.azure.account.oauth2.client.endpoint", f"https://login.microsoftonline.com/{storage_opts['tenant_id']}/oauth2/token") \
                .load(path)
            # Força a avaliação para detectar erros de leitura
            df.count()
            return df
        except Exception as e2:
            # Se ambos falharem, a tabela está corrompida – orienta regenerar
            raise Exception(
                f"Não foi possível ler a tabela Delta '{camada or 'raiz'}/{tabela}'. "
                f"Erro SDK: {e1}. Erro Spark: {e2}. "
                "Isso indica corrupção nos arquivos Parquet (ex.: tamanho zero). "
                "Execute novamente o notebook Silver para regenerar os logs de DQ."
            )



def gravar_delta_gold(df, nome_subtabela: str, storage_opts: dict, mode: str = "overwrite") -> bool:
    """
    Grava um DataFrame Spark como Delta físico dentro de gold/ecommerce_categorias/<nome_subtabela>.
    As tabelas Gold são escritas em modo overwrite (substituição completa a cada execução).
    Não há particionamento Hive nas Gold (são agregações pequenas prontas para BI).
    """
    path = get_delta_path_gold(nome_subtabela, storage_opts)
    try:
        pdf = df.toPandas()

        # Remove timezone de colunas datetime (requisito PyArrow)
        for col in pdf.columns:
            if pd.api.types.is_datetime64_any_dtype(pdf[col]):
                pdf[col] = pdf[col].dt.tz_localize(None)

        tabela_arrow = pa.Table.from_pandas(pdf, preserve_index=False)

        write_deltalake(
            table_or_uri=path,
            data=tabela_arrow,
            mode=mode,
            storage_options=storage_opts,
            schema_mode="overwrite",
        )
        print(f"  [Delta OK] {path} | {len(pdf)} linhas")
        return True
    except Exception as e:
        print(f"  [Delta ERRO] {path}: {e}")
        return False


# ─────────────────────────────────────────────────────────────────────────────
# Helper SQL Server
# Usa o conector nativo "sqlserver" disponível no Databricks Runtime —
# compatível com Serverless (não exige JAR externo).
# ─────────────────────────────────────────────────────────────────────────────

def salvar_sql_server(df, nome_tabela: str):
    """
    Espelha um DataFrame no Azure SQL Server via conector nativo Spark 'sqlserver'.
    O schema de destino é 'squad1'. Usa mode=overwrite (truncate + insert).
    """
    try:
        df.write \
            .format("sqlserver") \
            .option("host",     JDBC_HOSTNAME) \
            .option("port",     "1433") \
            .option("database", JDBC_DATABASE) \
            .option("dbtable",  f"squad1.{nome_tabela}") \
            .option("user",     JDBC_USERNAME) \
            .option("password", JDBC_PASSWORD) \
            .option("trustServerCertificate", "true") \
            .mode("overwrite") \
            .save()
        print(f"  [SQL OK] squad1.{nome_tabela}")
    except Exception as e:
        print(f"  [SQL ERRO] squad1.{nome_tabela}: {e}")


def publicar_gold(df, nome_subtabela: str, storage_opts: dict):
    """
    Wrapper unificado: grava no ADLS (Delta físico) e no SQL Server em sequência.
    Retorna o DataFrame para exibição opcional.
    """
    print(f"\n>>> Publicando: {nome_subtabela}")
    gravar_delta_gold(df, nome_subtabela, storage_opts)
    salvar_sql_server(df, nome_tabela=nome_subtabela)
    return df


## 3. Leitura das fontes (DQ Logs + Silver)

Antes de prosseguir, validamos se as tabelas de origem existem e contêm dados para a tabela alvo.
A função `ler_delta` agora é robusta contra arquivos Parquet corrompidos.

In [0]:
# ─── 3a. DQ Monitoring Logs (raiz do container) ──────────────────────────────
# ─── 3a. DQ Monitoring Logs (raiz do container) ──────────────────────────────
print("Lendo dq_monitoring_logs...")
df_dq_raw = ler_delta(camada=CAMADA_DQ, tabela=ENTIDADE_DQ, storage_opts=STORAGE_OPTIONS)

# Exibe um resumo das tabelas presentes nos logs (apenas para diagnóstico)
display(df_dq_raw.groupBy("tabela").count())

# Filtra apenas os logs relativos à tabela de interesse
df_dq = df_dq_raw.where(F.col("tabela") == TABELA_ALVO)

total_dq = df_dq.count()
print(f"  Registros de DQ para '{TABELA_ALVO}': {total_dq}")
if total_dq == 0:
    raise Exception(
        f"Nenhum log de DQ encontrado para tabela='{TABELA_ALVO}'. "
        "Execute o notebook Silver antes de rodar este notebook Gold."
    )

# ─── 3b. Silver ecommerce_categorias ─────────────────────────────────────────
print("\nLendo Silver ecommerce_categorias...")
df_silver = ler_delta(camada=CAMADA_SILVER, tabela=ENTIDADE_SILVER, storage_opts=STORAGE_OPTIONS)
total_silver = df_silver.count()
print(f"  Registros na Silver: {total_silver}")
if total_silver == 0:
    raise Exception(
        f"A tabela Silver '{CAMADA_SILVER}/{ENTIDADE_SILVER}' está vazia. "
        "Certifique-se de que o notebook Silver foi executado e gerou dados."
    )

# Preview
display(df_dq.limit(5))
display(df_silver.limit(5))

## 4. Gold 1 — `gold_ecommerce_categorias_dq_resumo_por_regra`

**Objetivo BI**: monitorar a % de falha de cada regra de DQ por dia.
Permite identificar degradações de qualidade regra a regra ao longo do tempo.

Colunas geradas:
- `data_execucao` — data (YYYY-MM-DD) derivada de `timestamp_execucao`
- `regra` — nome da regra
- `severidade` — Critica / Aviso
- `status` — PASS / FAIL (mais frequente do dia)
- `total_execucoes` — quantidade de registros de log para essa regra nesse dia
- `total_falhos` — soma de `qtd_registros_falhos`
- `total_registros` — soma de `qtd_registros_total`
- `pct_falha` — % de registros falhos sobre o total
- `gold_generated_at` — timestamp de geração desta tabela Gold


In [0]:
df_gold_regra = (
    df_dq
    .withColumn("data_execucao", F.to_date(F.col("timestamp_execucao")))
    .groupBy("data_execucao", "regra", "severidade")
    .agg(
        F.count("*").cast("long").alias("total_execucoes"),
        F.sum("qtd_registros_falhos").cast("long").alias("total_falhos"),
        F.sum("qtd_registros_total").cast("long").alias("total_registros"),
        # status consolidado: se houve pelo menos um FAIL no dia, dia é FAIL
        F.max(F.when(F.col("status") == "FAIL", 1).otherwise(0))
         .alias("_teve_fail")
    )
    .withColumn(
        "pct_falha",
        F.round(
            F.when(F.col("total_registros") > 0,
                   (F.col("total_falhos") / F.col("total_registros")) * 100
            ).otherwise(F.lit(0.0)),
            4
        )
    )
    .withColumn("status_dia", F.when(F.col("_teve_fail") == 1, "FAIL").otherwise("PASS"))
    .drop("_teve_fail")
    .withColumn("gold_generated_at", F.current_timestamp())
    .orderBy("data_execucao", "regra")
)

print(f"Linhas geradas: {df_gold_regra.count()}")
display(df_gold_regra)


## 5. Gold 2 — `gold_ecommerce_categorias_dq_resumo_por_tabela`

**Objetivo BI**: monitorar a % de registros limpos (válidos) por tabela por hora.
Útil para dashboards de saúde em tempo quase-real e alertas de SLA.

Colunas geradas:
- `data_hora` — janela horária (YYYY-MM-DD HH:00:00)
- `tabela`
- `total_registros_avaliados`
- `total_registros_limpos` — registros sem nenhuma falha em qualquer regra
- `pct_limpos` — % de registros limpos sobre o total avaliado
- `qtd_regras_com_falha` — quantas regras distintas registraram falha nessa hora
- `gold_generated_at`


In [0]:
# Para calcular registros limpos por hora, agregamos os logs por (tabela, hora, arquivo).
# Um arquivo é considerado "limpo" quando TODAS as regras passaram (status == PASS) para ele.

df_arquivos_hora = (
    df_dq
    .withColumn("data_hora",
        F.date_trunc("hour", F.col("timestamp_execucao")))
    .groupBy("tabela", "data_hora", "arquivo_origem")
    .agg(
        F.max("qtd_registros_total").alias("qtd_registros"),
        F.sum(F.when(F.col("status") == "FAIL", 1).otherwise(0)).alias("regras_que_falharam"),
    )
    .withColumn("arquivo_limpo",
        F.when(F.col("regras_que_falharam") == 0, 1).otherwise(0))
)

df_gold_tabela = (
    df_arquivos_hora
    .groupBy("tabela", "data_hora")
    .agg(
        F.sum("qtd_registros").cast("long").alias("total_registros_avaliados"),
        F.sum(
            F.when(F.col("arquivo_limpo") == 1, F.col("qtd_registros")).otherwise(0)
        ).cast("long").alias("total_registros_limpos"),
        F.sum(F.col("regras_que_falharam")).cast("long").alias("soma_falhas_por_arquivo"),
        # quantas regras distintas falharam nessa hora (aproximação via soma)
        F.count(F.when(F.col("arquivo_limpo") == 0, 1)).alias("arquivos_com_falha"),
    )
    .withColumn(
        "pct_limpos",
        F.round(
            F.when(F.col("total_registros_avaliados") > 0,
                   (F.col("total_registros_limpos") / F.col("total_registros_avaliados")) * 100
            ).otherwise(F.lit(0.0)),
            4
        )
    )
    .drop("soma_falhas_por_arquivo")
    .withColumn("gold_generated_at", F.current_timestamp())
    .orderBy("tabela", "data_hora")
)

print(f"Linhas geradas: {df_gold_tabela.count()}")
display(df_gold_tabela)


## 6. Gold 3 — `gold_ecommerce_categorias_dq_tendencia_diaria`

**Objetivo BI**: acompanhar o **score de qualidade** (0–100) por dia e sua variação (Δ)
em relação ao dia anterior. Ideal para gráficos de tendência e alertas de regressão.

Colunas geradas:
- `data_execucao`
- `score_qualidade` — 100 × (1 – pct_falha_média_diária/100)
- `pct_falha_media_dia` — média de % de falha entre todas as regras do dia
- `total_registros_dia`
- `total_falhos_dia`
- `qtd_regras_avaliadas`
- `qtd_regras_com_falha` — regras com ao menos 1 registro falho no dia
- `delta_score` — variação do score vs dia anterior (NULL no primeiro dia)
- `tendencia` — MELHORA / PIORA / ESTÁVEL / PRIMEIRO_DIA
- `gold_generated_at`


In [0]:
# Agrega por dia todos os logs
df_diario = (
    df_dq
    .withColumn("data_execucao", F.to_date(F.col("timestamp_execucao")))
    .groupBy("data_execucao")
    .agg(
        F.sum("qtd_registros_total").cast("long").alias("total_registros_dia"),
        F.sum("qtd_registros_falhos").cast("long").alias("total_falhos_dia"),
        F.countDistinct("regra").alias("qtd_regras_avaliadas"),
        F.sum(F.when(F.col("status") == "FAIL", 1).otherwise(0))
         .cast("long").alias("qtd_registros_fail_log"),
        # Média de pct_falha por regra no dia
        F.avg(
            F.when(F.col("qtd_registros_total") > 0,
                   (F.col("qtd_registros_falhos") / F.col("qtd_registros_total")) * 100
            ).otherwise(F.lit(0.0))
        ).alias("pct_falha_media_dia"),
    )
    .withColumn("qtd_regras_com_falha", F.col("qtd_registros_fail_log"))
    .drop("qtd_registros_fail_log")
    .withColumn(
        "score_qualidade",
        F.round(
            F.greatest(F.lit(0.0),
                       F.lit(100.0) - F.col("pct_falha_media_dia")),
            4
        )
    )
)

# Janela temporal para calcular delta_score
w_dia = Window.orderBy("data_execucao")

df_gold_tendencia = (
    df_diario
    .withColumn("score_anterior", F.lag("score_qualidade", 1).over(w_dia))
    .withColumn(
        "delta_score",
        F.round(F.col("score_qualidade") - F.col("score_anterior"), 4)
    )
    .withColumn(
        "tendencia",
        F.when(F.col("score_anterior").isNull(),          "PRIMEIRO_DIA")
         .when(F.col("delta_score") > 0.5,                "MELHORA")
         .when(F.col("delta_score") < -0.5,               "PIORA")
         .otherwise("ESTAVEL")
    )
    .drop("score_anterior")
    .withColumn("pct_falha_media_dia", F.round(F.col("pct_falha_media_dia"), 4))
    .withColumn("gold_generated_at", F.current_timestamp())
    .orderBy("data_execucao")
)

print(f"Linhas geradas: {df_gold_tendencia.count()}")
display(df_gold_tendencia)


## 7. Gold 4 — `gold_ecommerce_categorias_dq_regras_criticas`

**Objetivo BI**: lista consolidada das regras cujo percentual de falha acumulado
supera o threshold de **5%** (variável `THRESHOLD_FALHA_CRITICA`), indicando problemas recorrentes
que merecem atenção imediata da equipe de Data Quality.

Colunas geradas:
- `regra`, `severidade`
- `total_execucoes_historico`
- `total_falhos_historico`, `total_registros_historico`
- `pct_falha_historico` — % acumulada de falha
- `dias_com_falha` — quantos dias distintos essa regra falhou
- `primeiro_registro_falha`, `ultimo_registro_falha`
- `status_alerta` — CRITICO / ATENCAO / OK
- `threshold_pct` — valor do threshold usado
- `gold_generated_at`


In [0]:
df_gold_criticas = (
    df_dq
    .withColumn("data_execucao", F.to_date(F.col("timestamp_execucao")))
    .groupBy("regra", "severidade")
    .agg(
        F.count("*").cast("long").alias("total_execucoes_historico"),
        F.sum("qtd_registros_falhos").cast("long").alias("total_falhos_historico"),
        F.sum("qtd_registros_total").cast("long").alias("total_registros_historico"),
        F.countDistinct(
            F.when(F.col("status") == "FAIL", F.col("data_execucao"))
        ).alias("dias_com_falha"),
        F.min("timestamp_execucao").alias("primeiro_registro_falha"),
        F.max("timestamp_execucao").alias("ultimo_registro_falha"),
    )
    .withColumn(
        "pct_falha_historico",
        F.round(
            F.when(F.col("total_registros_historico") > 0,
                   (F.col("total_falhos_historico") / F.col("total_registros_historico")) * 100
            ).otherwise(F.lit(0.0)),
            4
        )
    )
    .withColumn(
        "status_alerta",
        F.when(
            (F.col("pct_falha_historico") >= THRESHOLD_FALHA_CRITICA) &
            (F.col("severidade") == "Critica"),
            "CRITICO"
        )
        .when(F.col("pct_falha_historico") >= THRESHOLD_FALHA_CRITICA, "ATENCAO")
        .otherwise("OK")
    )
    .withColumn("threshold_pct", F.lit(THRESHOLD_FALHA_CRITICA))
    .withColumn("gold_generated_at", F.current_timestamp())
    .orderBy(F.col("pct_falha_historico").desc())
)

print(f"Linhas geradas: {df_gold_criticas.count()}")
display(df_gold_criticas)


## 8. Gold 5 — `gold_ecommerce_categorias_snapshot_validas`

**Objetivo BI**: visão analítica das categorias Silver **válidas** (`silver_linha_valida = true`),
enriquecida com métricas de hierarquia e flags de DQ por categoria.
Base para análises de cobertura de catálogo no Looker.

Colunas geradas a partir da Silver:
- Colunas de negócio: `id_categoria`, `nome_categoria`, `id_categoria_pai`, `nivel_hierarquico`
- `tipo_categoria` — RAIZ / SUBCATEGORIA
- `qtd_filhos_diretos` — para categorias raiz: quantas subcategorias diretas existem
- Contagem das regras que falharam por registro (`qtd_regras_falhas`)
- `score_registro` — 100 × (1 – qtd_regras_falhas/10)
- `silver_processed_at`, `gold_generated_at`

> Apenas registros com `silver_linha_valida = true` compõem esta tabela.


In [0]:
# Seleciona colunas relevantes da Silver e calcula métricas de hierarquia
flags_regra = [c for c in df_silver.columns if c.startswith("r") and c.endswith("_falhou")]
n_regras    = max(len(flags_regra), 1)

# Contagem de filhos diretos por categoria (número de subcategorias diretas)
# Usamos a Silver completa (não apenas as válidas) para contar todos os filhos,
# independentemente de sua validade, pois isso reflete a estrutura real.
df_filhos = (
    df_silver
    .where(F.col("id_categoria_pai_str").isNotNull())
    .groupBy(F.col("id_categoria_pai_str").alias("id_cat_ref"))
    .agg(F.count("*").alias("qtd_filhos_diretos"))
)

# Expressão para somar regras que falharam
if flags_regra:
    expr_qtd_falhas = reduce(
        lambda a, b: a + b,
        [F.when(F.col(c), 1).otherwise(0).cast("int") for c in flags_regra]
    )
else:
    expr_qtd_falhas = F.lit(0)

df_gold_snapshot = (
    df_silver
    .where(F.col("silver_linha_valida") == True)
    .select(
        F.col("id_categoria").cast("string").alias("id_categoria"),
        F.col("nome_categoria").alias("nome_categoria"),
        F.col("id_categoria_pai").cast("string").alias("id_categoria_pai"),
        F.col("id_categoria_str"),
        F.col("id_categoria_pai_str"),
        F.col("silver_processed_at"),
        *[F.col(c) for c in flags_regra]
    )
    # Nível hierárquico: 0 = raiz, 1 = subcategoria
    .withColumn(
        "nivel_hierarquico",
        F.when(F.col("id_categoria_pai_str").isNull(), F.lit(0)).otherwise(F.lit(1))
    )
    .withColumn(
        "tipo_categoria",
        F.when(F.col("id_categoria_pai_str").isNull(), "RAIZ").otherwise("SUBCATEGORIA")
    )
    # Junta qtd_filhos_diretos
    .join(df_filhos, F.col("id_categoria_str") == F.col("id_cat_ref"), how="left")
    .withColumn("qtd_filhos_diretos",
        F.coalesce(F.col("qtd_filhos_diretos"), F.lit(0)).cast("long"))
    .drop("id_cat_ref")
    # Score por registro
    .withColumn("qtd_regras_falhas", expr_qtd_falhas.cast("int"))
    .withColumn(
        "score_registro",
        F.round(F.lit(100.0) - (F.col("qtd_regras_falhas") / F.lit(n_regras)) * 100.0, 2)
    )
    # Remove flags individuais (detalhe interno da Silver)
    .drop(*flags_regra)
    .drop("id_categoria_str", "id_categoria_pai_str")
    .withColumn("gold_generated_at", F.current_timestamp())
    .orderBy("tipo_categoria", "id_categoria")
)

print(f"Categorias válidas no snapshot: {df_gold_snapshot.count()}")
display(df_gold_snapshot)


## 9. Publicação: ADLS (Delta físico) + Azure SQL Server

Cada tabela Gold é publicada no Delta e espelhada no SQL Server.

In [0]:
print("=" * 70)
print("INICIANDO PUBLICAÇÃO DA CAMADA GOLD")
print("=" * 70)

# 1. DQ resumo por regra por dia
publicar_gold(df_gold_regra,      TABELA_GOLD_REGRA,     STORAGE_OPTIONS)

# 2. DQ resumo por tabela por hora
publicar_gold(df_gold_tabela,     TABELA_GOLD_TABELA,    STORAGE_OPTIONS)

# 3. Tendência diária de score de qualidade
publicar_gold(df_gold_tendencia,  TABELA_GOLD_TENDENCIA, STORAGE_OPTIONS)

# 4. Regras críticas (acima do threshold)
publicar_gold(df_gold_criticas,   TABELA_GOLD_CRITICAS,  STORAGE_OPTIONS)

# 5. Snapshot analítico das categorias válidas
publicar_gold(df_gold_snapshot,   TABELA_GOLD_SNAPSHOT,  STORAGE_OPTIONS)

print("\n" + "=" * 70)
print(f"CAMADA GOLD FINALIZADA — RUN_ID: {RUN_ID}")
print(f"Tabelas Delta em: abfss://{CONTAINER_SQUAD1}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/{CAMADA_GOLD}/{ENTIDADE_GOLD}/")
print(f"Tabelas SQL Server em schema: squad1.*")
print("=" * 70)


## 10. Validação final — contagem de linhas por tabela Gold

In [0]:
tabelas_gold = {
    TABELA_GOLD_REGRA:      df_gold_regra,
    TABELA_GOLD_TABELA:     df_gold_tabela,
    TABELA_GOLD_TENDENCIA:  df_gold_tendencia,
    TABELA_GOLD_CRITICAS:   df_gold_criticas,
    TABELA_GOLD_SNAPSHOT:   df_gold_snapshot,
}

resumo = [(nome, df.count()) for nome, df in tabelas_gold.items()]
df_resumo = spark.createDataFrame(resumo, ["tabela_gold", "qtd_linhas"])
display(df_resumo)

print("\n✅ Validação concluída.")


## 11. Graficos de resultados - Análise em Data Quality

In [0]:
# ─── Gráfico 1: Resumo de falha por regra ─────────────────────────────────────
# Leitura da tabela Gold: gold_ecommerce_categorias_dq_resumo_por_regra
# Gráfico: barras horizontais com % de falha por regra, ordenado decrescente.

import matplotlib.pyplot as plt
import pandas as pd

# 1. Parâmetros da tabela
camada_gold = CAMADA_GOLD
tabela_gold_regra = f"{ENTIDADE_GOLD}/{TABELA_GOLD_REGRA}"  # "ecommerce_categorias/gold_ecommerce_categorias_dq_resumo_por_regra"

# 2. Verifica se a tabela existe usando delta_existe (já definida)
if delta_existe(camada_gold, tabela_gold_regra, STORAGE_OPTIONS):
    # 3. Lê os dados
    df_gold_regra = ler_delta(camada_gold, tabela_gold_regra, STORAGE_OPTIONS)
    
    # 4. Converte para Pandas para plotagem
    pdf_regra = df_gold_regra.toPandas()
    
    if not pdf_regra.empty:
        # 5. Ordena por pct_falha decrescente
        pdf_regra_sorted = pdf_regra.sort_values("pct_falha", ascending=False)
        
        # 6. Define cores por severidade
        cores = {"Critica": "#d9534f", "Aviso": "#f0ad4e"}
        pdf_regra_sorted["cor"] = pdf_regra_sorted["severidade"].map(cores)
        
        # 7. Cria o gráfico de barras horizontais
        fig, ax = plt.subplots(figsize=(10, 6))
        bars = ax.barh(
            pdf_regra_sorted["regra"],
            pdf_regra_sorted["pct_falha"],
            color=pdf_regra_sorted["cor"]
        )
        
        # 8. Adiciona rótulos e título
        ax.set_xlabel("% de Falha", fontsize=12)
        ax.set_title("Resumo de Falha por Regra de DQ", fontsize=14, fontweight="bold")
        ax.invert_yaxis()  # Maior % no topo
        
        # 9. Adiciona valores ao lado das barras
        for bar in bars:
            width = bar.get_width()
            ax.text(
                width + 1,
                bar.get_y() + bar.get_height()/2,
                f"{width:.1f}%",
                va="center",
                fontsize=10
            )
        
        # 10. Ajusta layout e exibe
        plt.tight_layout()
        display(fig)
    else:
        print("⚠️ A tabela gold_ecommerce_categorias_dq_resumo_por_regra está vazia.")
else:
    print("⚠️ Tabela gold_ecommerce_categorias_dq_resumo_por_regra não encontrada.")

In [0]:
# ─── Gráfico 2: Resumo de registros limpos por tabela e hora ─────────────────
# Leitura da tabela Gold: gold_ecommerce_categorias_dq_resumo_por_tabela
# Gráfico: linhas com a evolução do percentual de registros limpos ao longo do tempo.

import matplotlib.pyplot as plt
import pandas as pd

# 1. Parâmetros da tabela
camada_gold = CAMADA_GOLD
tabela_gold_tabela = f"{ENTIDADE_GOLD}/{TABELA_GOLD_TABELA}"

# 2. Verifica se a tabela existe
if delta_existe(camada_gold, tabela_gold_tabela, STORAGE_OPTIONS):
    # 3. Lê os dados
    df_gold_tabela = ler_delta(camada_gold, tabela_gold_tabela, STORAGE_OPTIONS)
    
    # 4. Converte para Pandas
    pdf_tabela = df_gold_tabela.toPandas()
    
    if not pdf_tabela.empty:
        # 5. Ordena por data_hora
        pdf_tabela_sorted = pdf_tabela.sort_values("data_hora")
        
        # 6. Cria o gráfico de linhas
        fig, ax = plt.subplots(figsize=(12, 6))
        
        # Plota a porcentagem de limpos
        ax.plot(
            pdf_tabela_sorted["data_hora"],
            pdf_tabela_sorted["pct_limpos"],
            marker="o",
            linestyle="-",
            linewidth=2,
            markersize=8,
            color="#2c3e50",
            label="% Registros Limpos"
        )
        
        # 7. Adiciona título e rótulos
        ax.set_xlabel("Data/Hora", fontsize=12)
        ax.set_ylabel("% de Registros Limpos", fontsize=12)
        ax.set_title("Evolução da Qualidade da Tabela (Registros Limpos)", fontsize=14, fontweight="bold")
        ax.grid(True, linestyle="--", alpha=0.6)
        ax.legend(loc="best")
        
        # 8. Adiciona rótulos de valor nos pontos
        for i, row in pdf_tabela_sorted.iterrows():
            ax.annotate(
                f"{row['pct_limpos']:.1f}%",
                (row["data_hora"], row["pct_limpos"]),
                textcoords="offset points",
                xytext=(0, 10),
                ha="center",
                fontsize=9
            )
        
        # 9. Ajusta layout e exibe
        plt.xticks(rotation=45)
        plt.tight_layout()
        display(fig)
    else:
        print("⚠️ A tabela gold_ecommerce_categorias_dq_resumo_por_tabela está vazia.")
else:
    print("⚠️ Tabela gold_ecommerce_categorias_dq_resumo_por_tabela não encontrada.")

In [0]:
# ─── Gráfico 3: Tendência diária do score de qualidade ────────────────────────
# Leitura da tabela Gold: gold_ecommerce_categorias_dq_tendencia_diaria
# Gráfico: linhas com o score de qualidade ao longo dos dias, com cores por tendência.

import matplotlib.pyplot as plt
import pandas as pd

# 1. Parâmetros da tabela
camada_gold = CAMADA_GOLD
tabela_gold_tendencia = f"{ENTIDADE_GOLD}/{TABELA_GOLD_TENDENCIA}"

# 2. Verifica se a tabela existe
if delta_existe(camada_gold, tabela_gold_tendencia, STORAGE_OPTIONS):
    # 3. Lê os dados
    df_gold_tendencia = ler_delta(camada_gold, tabela_gold_tendencia, STORAGE_OPTIONS)
    
    # 4. Converte para Pandas
    pdf_tendencia = df_gold_tendencia.toPandas()
    
    if not pdf_tendencia.empty:
        # 5. Ordena por data_execucao
        pdf_tendencia_sorted = pdf_tendencia.sort_values("data_execucao")
        
        # 6. Define cores por tendência
        cores_tendencia = {
            "MELHORA": "#2ecc71",   # verde
            "PIORA": "#e74c3c",      # vermelho
            "ESTAVEL": "#95a5a6",    # cinza
            "PRIMEIRO_DIA": "#3498db" # azul
        }
        pdf_tendencia_sorted["cor"] = pdf_tendencia_sorted["tendencia"].map(cores_tendencia)
        
        # 7. Cria figura com dois subplots: score e delta
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
        
        # --- Subplot 1: Score de qualidade ---
        # Plota barras com cores por tendência
        bars = ax1.bar(
            pdf_tendencia_sorted["data_execucao"],
            pdf_tendencia_sorted["score_qualidade"],
            color=pdf_tendencia_sorted["cor"],
            alpha=0.7,
            edgecolor="black"
        )
        
        # Adiciona rótulos de valor nas barras
        for bar, score in zip(bars, pdf_tendencia_sorted["score_qualidade"]):
            ax1.text(
                bar.get_x() + bar.get_width()/2,
                bar.get_height() + 1,
                f"{score:.1f}",
                ha="center",
                va="bottom",
                fontsize=10,
                fontweight="bold"
            )
        
        ax1.set_ylabel("Score de Qualidade (0-100)", fontsize=12)
        ax1.set_title("Evolução Diária do Score de Qualidade", fontsize=14, fontweight="bold")
        ax1.grid(True, linestyle="--", alpha=0.5)
        ax1.set_ylim(0, 110)  # Deixa espaço para os rótulos
        
        # --- Subplot 2: Delta do score ---
        # Gráfico de barras para delta_score
        cores_delta = [
            "#2ecc71" if d > 0 else "#e74c3c" if d < 0 else "#95a5a6"
            for d in pdf_tendencia_sorted["delta_score"].fillna(0)
        ]
        ax2.bar(
            pdf_tendencia_sorted["data_execucao"],
            pdf_tendencia_sorted["delta_score"].fillna(0),
            color=cores_delta,
            alpha=0.7,
            edgecolor="black"
        )
        
        ax2.axhline(y=0, color="black", linestyle="-", linewidth=0.8)
        ax2.set_xlabel("Data", fontsize=12)
        ax2.set_ylabel("Variação do Score (Δ)", fontsize=12)
        ax2.set_title("Variação do Score em Relação ao Dia Anterior", fontsize=14, fontweight="bold")
        ax2.grid(True, linestyle="--", alpha=0.5)
        
        # Adiciona rótulos de tendência no subplot 2
        for i, row in pdf_tendencia_sorted.iterrows():
            if pd.notna(row["delta_score"]):
                ax2.annotate(
                    f"{row['tendencia']}\n{row['delta_score']:+.1f}",
                    (row["data_execucao"], row["delta_score"]),
                    textcoords="offset points",
                    xytext=(0, 10 if row["delta_score"] >= 0 else -20),
                    ha="center",
                    fontsize=9,
                    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="gray", alpha=0.7)
                )
        
        # 8. Ajusta layout e exibe
        plt.xticks(rotation=45)
        plt.tight_layout()
        display(fig)
    else:
        print("⚠️ A tabela gold_ecommerce_categorias_dq_tendencia_diaria está vazia.")
else:
    print("⚠️ Tabela gold_ecommerce_categorias_dq_tendencia_diaria não encontrada.")

In [0]:
# ─── Gráfico 3: Tendência diária de qualidade ─────────────────────────────────
# Leitura da tabela Gold: gold_ecommerce_categorias_dq_tendencia_diaria
# Gráfico combinado: linha para o score de qualidade e barras para a variação (delta).

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 1. Parâmetros da tabela
camada_gold = CAMADA_GOLD
tabela_gold_tendencia = f"{ENTIDADE_GOLD}/{TABELA_GOLD_TENDENCIA}"

# 2. Verifica se a tabela existe
if delta_existe(camada_gold, tabela_gold_tendencia, STORAGE_OPTIONS):
    # 3. Lê os dados
    df_gold_tendencia = ler_delta(camada_gold, tabela_gold_tendencia, STORAGE_OPTIONS)
    
    # 4. Converte para Pandas
    pdf_tendencia = df_gold_tendencia.toPandas()
    
    if not pdf_tendencia.empty:
        # 5. Ordena por data_execucao
        pdf_tendencia_sorted = pdf_tendencia.sort_values("data_execucao")
        
        # 6. Cria o gráfico combinado (duplo eixo)
        fig, ax1 = plt.subplots(figsize=(12, 6))
        
        # 6a. Eixo principal: score de qualidade (linha)
        ax1.plot(
            pdf_tendencia_sorted["data_execucao"],
            pdf_tendencia_sorted["score_qualidade"],
            marker="o",
            linestyle="-",
            linewidth=2,
            markersize=8,
            color="#2980b9",
            label="Score de Qualidade"
        )
        ax1.set_xlabel("Data", fontsize=12)
        ax1.set_ylabel("Score de Qualidade (0-100)", fontsize=12, color="#2980b9")
        ax1.tick_params(axis="y", labelcolor="#2980b9")
        ax1.grid(True, linestyle="--", alpha=0.3)
        
        # 6b. Eixo secundário: delta_score (barras)
        ax2 = ax1.twinx()
        cores_delta = [
            "#27ae60" if d > 0 else "#e74c3c" if d < 0 else "#95a5a6"
            for d in pdf_tendencia_sorted["delta_score"].fillna(0)
        ]
        ax2.bar(
            pdf_tendencia_sorted["data_execucao"],
            pdf_tendencia_sorted["delta_score"].fillna(0),
            width=0.6,
            color=cores_delta,
            alpha=0.6,
            label="Variação (Δ)"
        )
        ax2.set_ylabel("Variação do Score (Δ)", fontsize=12, color="#2c3e50")
        ax2.tick_params(axis="y", labelcolor="#2c3e50")
        ax2.axhline(y=0, color="black", linestyle="-", linewidth=0.8, alpha=0.5)
        
        # 7. Adiciona título e legendas combinadas
        ax1.set_title("Tendência Diária de Qualidade", fontsize=14, fontweight="bold")
        
        # Combina legendas dos dois eixos
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc="best")
        
        # 8. Ajusta layout e exibe
        plt.xticks(rotation=45)
        plt.tight_layout()
        display(fig)
    else:
        print("⚠️ A tabela gold_ecommerce_categorias_dq_tendencia_diaria está vazia.")
else:
    print("⚠️ Tabela gold_ecommerce_categorias_dq_tendencia_diaria não encontrada.")

In [0]:
# ─── Gráfico 5: Snapshot de categorias válidas ────────────────────────────────
# Leitura da tabela Gold: gold_ecommerce_categorias_snapshot_validas
# Gráfico: barras com a distribuição de categorias por tipo (RAIZ vs SUBCATEGORIA)
# e score médio de qualidade por tipo.

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from io import BytesIO
from azure.storage.filedatalake import DataLakeServiceClient
from azure.identity import ClientSecretCredential
from pyspark.sql.types import StructType

# 1. Garantir que temos o cliente do ADLS
if 'file_system_squad1' not in dir():
    credential = ClientSecretCredential(
        tenant_id=TENANT_ID,
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET
    )
    service_client = DataLakeServiceClient(
        account_url=f"https://{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net",
        credential=credential
    )
    file_system_squad1 = service_client.get_file_system_client(file_system=CONTAINER_SQUAD1)
    print("✅ Cliente ADLS criado localmente.")

# 2. Parâmetros da tabela
pasta_snapshot = f"{CAMADA_GOLD}/{ENTIDADE_GOLD}/{TABELA_GOLD_SNAPSHOT}"
print(f"🔍 Verificando tabela em: squad1/{pasta_snapshot}")

# 3. Função de verificação robusta
def tabela_existe_adls(pasta_raiz):
    try:
        log_paths = list(file_system_squad1.get_paths(path=f"{pasta_raiz}/_delta_log", recursive=False))
        if log_paths:
            print(f"   ✅ _delta_log encontrado com {len(log_paths)} arquivo(s)")
            return True
    except Exception as e:
        print(f"   ⚠️ Erro ao listar _delta_log: {e}")
    
    try:
        all_paths = list(file_system_squad1.get_paths(path=pasta_raiz, recursive=True))
        for p in all_paths:
            if not p.is_directory and p.name.endswith(".parquet"):
                print(f"   ✅ Parquet encontrado: {p.name}")
                return True
    except Exception as e:
        print(f"   ⚠️ Erro ao listar Parquet: {e}")
    
    return False

# 4. Função para ler todos os Parquet de uma pasta
def ler_parquets_adls(pasta_raiz):
    paths = list(file_system_squad1.get_paths(path=pasta_raiz, recursive=True))
    arquivos_parquet = [p.name for p in paths if not p.is_directory and p.name.endswith(".parquet")]
    if not arquivos_parquet:
        print("   ⚠️ Nenhum arquivo Parquet encontrado.")
        return spark.createDataFrame([], schema=StructType([]))
    
    print(f"   Lendo {len(arquivos_parquet)} arquivo(s) Parquet de '{pasta_raiz}'...")
    dfs = []
    for arq in arquivos_parquet:
        fc = file_system_squad1.get_file_client(arq)
        raw = fc.download_file().readall()
        pdf = pd.read_parquet(BytesIO(raw))
        dfs.append(pdf)
    
    pdf_total = pd.concat(dfs, ignore_index=True)
    for col in pdf_total.columns:
        if pd.api.types.is_datetime64_any_dtype(pdf_total[col]):
            try:
                pdf_total[col] = pdf_total[col].dt.tz_localize(None)
            except TypeError:
                pdf_total[col] = pdf_total[col].dt.tz_convert(None)
    return spark.createDataFrame(pdf_total)

# 5. Verifica se a tabela existe
if tabela_existe_adls(pasta_snapshot):
    print("✅ Tabela encontrada! Lendo dados...")
    df_gold_snapshot = ler_parquets_adls(pasta_snapshot)
    pdf_snapshot = df_gold_snapshot.toPandas()
    
    if not pdf_snapshot.empty:
        agg_snapshot = pdf_snapshot.groupby("tipo_categoria").agg(
            qtd_categorias=("id_categoria", "count"),
            score_medio=("score_registro", "mean")
        ).reset_index()
        
        fig, ax1 = plt.subplots(figsize=(10, 6))
        x = range(len(agg_snapshot))
        bars = ax1.bar(
            x,
            agg_snapshot["qtd_categorias"],
            width=0.4,
            color=["#3498db", "#2ecc71"],
            label="Quantidade de Categorias"
        )
        ax1.set_xlabel("Tipo de Categoria", fontsize=12)
        ax1.set_ylabel("Quantidade de Categorias", fontsize=12, color="#3498db")
        ax1.tick_params(axis="y", labelcolor="#3498db")
        ax1.set_xticks(x)
        ax1.set_xticklabels(agg_snapshot["tipo_categoria"])
        
        for bar in bars:
            height = bar.get_height()
            ax1.annotate(
                f"{int(height)}",
                xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 5),
                textcoords="offset points",
                ha="center",
                fontsize=10,
                fontweight="bold"
            )
        
        ax2 = ax1.twinx()
        ax2.plot(
            x,
            agg_snapshot["score_medio"],
            marker="o",
            linestyle="-",
            linewidth=2,
            markersize=8,
            color="#e74c3c",
            label="Score Médio"
        )
        ax2.set_ylabel("Score Médio de Qualidade", fontsize=12, color="#e74c3c")
        ax2.tick_params(axis="y", labelcolor="#e74c3c")
        
        for i, row in agg_snapshot.iterrows():
            ax2.annotate(
                f"{row['score_medio']:.1f}",
                (i, row["score_medio"]),
                textcoords="offset points",
                xytext=(0, 10),
                ha="center",
                fontsize=9,
                color="#e74c3c"
            )
        
        plt.title("Snapshot de Categorias Válidas", fontsize=14, fontweight="bold")
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
        plt.tight_layout()
        display(fig)
    else:
        print("ℹ️ A tabela gold_ecommerce_categorias_snapshot_validas existe, mas está vazia (nenhuma categoria válida encontrada).")
else:
    print("⚠️ Tabela gold_ecommerce_categorias_snapshot_validas não encontrada.")
    print(f"   Caminho verificado: squad1/{pasta_snapshot}")